# Data Exploration

Import data into a dataframe, explore

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))


from src.dataExtraction.extract import import_data


data_path = (project_root / "data/raw/BPI_Challenge_2013_incidents/BPI_Challenge_2013_incidents.xes")

df = import_data(str(data_path), ["impact","org:role"])

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:01<00:00, 5975.43it/s]


      org:group resource country organization country org:resource  \
0           V30           France                   fr     Frederic   
1           V30           France                   fr     Frederic   
2        V5 3rd           France                   fr     Frederic   
3        V5 3rd           France                   fr  Anne Claire   
4           V30           France                   fr  Anne Claire   
...         ...              ...                  ...          ...   
65528        C9           Brazil                   br      Lierson   
65529        C9           Brazil                   br      Lierson   
65530        C9           Brazil                   br      Lierson   
65531        C9           Brazil                   br      Lierson   
65532       N36              USA                   us         Matt   

      organization involved org:role concept:name  impact  product  \
0               Org line A2     A2_4     Accepted  Medium  PROD582   
1               Org

In [2]:
df.head()

,org:group,resource country,organization country,org:resource,organization involved,org:role,concept:name,impact,product,lifecycle:transition,time:timestamp,case:concept:name
0,V30,France,fr,Frederic,Org line A2,A2_4,Accepted,Medium,PROD582,In Progress,2010-03-31 16:59:42+00:00,1-364285768
1,V30,France,fr,Frederic,Org line A2,A2_4,Accepted,Medium,PROD582,In Progress,2010-03-31 17:00:56+00:00,1-364285768
2,V5 3rd,France,fr,Frederic,Org line A2,A2_5,Queued,Medium,PROD582,Awaiting Assignment,2010-03-31 17:45:48+00:00,1-364285768
3,V5 3rd,France,fr,Anne Claire,Org line A2,A2_5,Accepted,Medium,PROD582,In Progress,2010-04-06 16:44:07+00:00,1-364285768
4,V30,France,fr,Anne Claire,Org line A2,A2_4,Queued,Medium,PROD582,Awaiting Assignment,2010-04-06 16:44:38+00:00,1-364285768


In [3]:
df.columns

Index(['org:group', 'resource country', 'organization country', 'org:resource',
       'organization involved', 'org:role', 'concept:name', 'impact',
       'product', 'lifecycle:transition', 'time:timestamp',
       'case:concept:name'],
      dtype='str')

In [4]:
df.shape

(65533, 12)

In [5]:
df["case:concept:name"].nunique()

7554

In [6]:
df.groupby("case:concept:name").size().head()

case:concept:name
1-364285768    17
1-467153946    40
1-503573772    17
1-504538555    19
1-506071646    62
dtype: int64

In [7]:
df["lifecycle:transition"].unique()

<StringArray>
[          'In Progress',   'Awaiting Assignment',              'Resolved',
              'Assigned',                'Closed',           'Wait - User',
 'Wait - Implementation',                  'Wait',         'Wait - Vendor',
               'In Call',       'Wait - Customer',             'Unmatched',
             'Cancelled']
Length: 13, dtype: str

## Outcome definition (first version)
#### **Idea**: *median duration* of the case as the threshold value

In [8]:
import pandas as pd

# convert timestamps into datetime
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])
df.dtypes

org:group                                str
resource country                         str
organization country                     str
org:resource                             str
organization involved                    str
org:role                                 str
concept:name                             str
impact                                   str
product                                  str
lifecycle:transition                     str
time:timestamp           datetime64[us, UTC]
case:concept:name                        str
dtype: object

In [9]:
case_df = df.groupby("case:concept:name")["time:timestamp"].agg(["min", "max"])
case_df["duration"] = case_df["max"] - case_df["min"]
outcome_threshold = case_df["duration"].median()
outcome_threshold

Timedelta('7 days 13:10:44.500000')

#### After defining threshold value create outcome label

In [10]:
case_df["outcome"] = (case_df["duration"] < outcome_threshold).astype(int)
case_df["outcome"].head()

case:concept:name
1-364285768    0
1-467153946    0
1-503573772    0
1-504538555    0
1-506071646    0
Name: outcome, dtype: int64

## Test outcome function
The logic above was transfered into a function *compute_duration_outcomes* in *src.featureEngineering.outcomeLabelling*

In [12]:
from src.featureEngineering.outcome_labelling import compute_duration_outcomes

df = import_data(str(data_path), ["impact","org:role"])
case_df = compute_duration_outcomes(df)

case_df.head()

parsing log, completed traces :: 100%|██████████| 7554/7554 [00:01<00:00, 5838.57it/s]


      org:group resource country organization country org:resource  \
0           V30           France                   fr     Frederic   
1           V30           France                   fr     Frederic   
2        V5 3rd           France                   fr     Frederic   
3        V5 3rd           France                   fr  Anne Claire   
4           V30           France                   fr  Anne Claire   
...         ...              ...                  ...          ...   
65528        C9           Brazil                   br      Lierson   
65529        C9           Brazil                   br      Lierson   
65530        C9           Brazil                   br      Lierson   
65531        C9           Brazil                   br      Lierson   
65532       N36              USA                   us         Matt   

      organization involved org:role concept:name  impact  product  \
0               Org line A2     A2_4     Accepted  Medium  PROD582   
1               Org

,case:concept:name,case_start,case_end,duration,outcome
0,1-364285768,2010-03-31 16:59:42+00:00,2012-05-11 01:26:15+00:00,771 days 08:26:33,0
1,1-467153946,2011-01-31 11:12:22+00:00,2012-05-23 01:22:25+00:00,477 days 14:10:03,0
2,1-503573772,2011-02-24 16:17:46+00:00,2012-05-12 01:21:31+00:00,442 days 09:03:45,0
3,1-504538555,2011-02-28 14:13:33+00:00,2012-05-12 01:21:30+00:00,438 days 11:07:57,0
4,1-506071646,2011-03-07 10:42:08+00:00,2012-05-12 01:21:30+00:00,431 days 14:39:22,0


In [ ]:
case_df["outcome"].value_counts()

outcome
0    3777
1    3777
Name: count, dtype: int64